# Train Models

In [20]:
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, LSTM, Dense, Dropout
import os
import pickle
from concurrent.futures import ThreadPoolExecutor, as_completed

# Constants
start_date = '2019-01-01'
end_date = pd.Timestamp.today().strftime('%Y-%m-%d')
tickers = [
    'AAPL', 'MSFT', 'JPM', 'V', 'JNJ', 'WMT', 'PG', 'UNH', 'HD', 'INTC',
    'VZ', 'MRK', 'KO', 'DIS', 'CSCO', 'BA', 'MCD', 'IBM', 'GS', 'AXP',
    'NKE', 'TRV', 'CAT', 'MMM', 'WBA', 'DOW', 'AMGN', 'CRM', 'HON', 'AIG'
]

# Create directories
os.makedirs('models', exist_ok=True)
os.makedirs('scalers', exist_ok=True)

# Helper functions
def download_data(ticker):
    try:
        print(f"Downloading data for {ticker}")
        data = yf.download(ticker, start=start_date, end=end_date)
        if data.empty:
            print(f"No data found for {ticker}")
            return None
        return data
    except Exception as e:
        print(f"Error downloading data for {ticker}: {e}")
        return None

def preprocess_data(data):
    try:
        data['Volume'] = data['Volume'].replace(0, np.nan).fillna(method='ffill')
        data = data[['Close', 'Volume']]
        return data
    except Exception as e:
        print(f"Error preprocessing data: {e}")
        return None

def create_sequences(data, time_steps=20):
    sequences, targets = [], []
    try:
        for i in range(len(data) - time_steps):
            sequences.append(data[i:i+time_steps])
            targets.append(data[i+time_steps, 0])
    except Exception as e:
        print(f"Error creating sequences: {e}")
    return np.array(sequences), np.array(targets)

def create_model(input_shape, complexity):
    model = Sequential()
    for _ in range(complexity):
        model.add(Bidirectional(LSTM(units=50, return_sequences=True, input_shape=input_shape)))
        model.add(Dropout(0.2))
    model.add(Bidirectional(LSTM(units=50)))
    model.add(Dropout(0.2))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

def train_and_save_model(ticker, complexity, data, scaler, save_path_model, save_path_scaler):
    try:
        x_train, y_train = data
        model = create_model((x_train.shape[1], x_train.shape[2]), complexity)
        model.fit(x_train, y_train, epochs=100, batch_size=32, verbose=0)
        model.save(save_path_model)
        with open(save_path_scaler, 'wb') as f:
            pickle.dump(scaler, f)
        return model
    except Exception as e:
        print(f"Error training and saving model for {ticker}, complexity {complexity}: {e}")
        return None

def calculate_metrics(model, x_test, y_test):
    try:
        predictions = model.predict(x_test)
        rmse = np.sqrt(mean_squared_error(y_test, predictions))
        r2 = r2_score(y_test, predictions)
        sharpe_ratio = (predictions.mean() - 0.03) / predictions.std()
        return rmse, r2, sharpe_ratio
    except Exception as e:
        print(f"Error calculating metrics: {e}")
        return None, None, None

def process_ticker(ticker, data):
    try:
        data = preprocess_data(data)
        if data is None:
            return None

        scaler = StandardScaler()
        data_scaled = scaler.fit_transform(data)

        sequences, targets = create_sequences(data_scaled)
        if len(sequences) == 0:
            print(f"Not enough data to create sequences for {ticker}")
            return None

        # Use a time-based split to prevent data leakage
        split_date = pd.Timestamp('2024-01-01')
        split_index = np.where(data.index >= split_date)[0][0] - 20  # Adjust for sequence length

        x_train, y_train = sequences[:split_index], targets[:split_index]
        x_test, y_test = sequences[split_index:], targets[split_index:]

        results = []
        complexities = [3, 4, 5]  # Minimum complexity of 3 layers
        for complexity in complexities:
            model_path = f'models/{ticker}_complexity_{complexity}.keras'
            scaler_path = f'scalers/{ticker}_scaler_{complexity}.pkl'

            model = train_and_save_model(ticker, complexity, (x_train, y_train), scaler, model_path, scaler_path)
            if model is None:
                continue
            rmse, r2, sharpe_ratio = calculate_metrics(model, x_test, y_test)
            if rmse is None or r2 < 0.5:
                continue

            results.append((ticker, complexity, rmse, r2, sharpe_ratio))
        
        return results

    except Exception as e:
        print(f"Error processing {ticker}: {e}")
        return None

def main():
    # Step 1: Download all data sequentially
    data_dict = {}
    for ticker in tickers:
        data = download_data(ticker)
        if data is not None:
            data_dict[ticker] = data
    
    # Step 2: Process data in parallel
    results = []
    with ThreadPoolExecutor(max_workers=5) as executor:
        futures = {executor.submit(process_ticker, ticker, data): ticker for ticker, data in data_dict.items()}
        for future in as_completed(futures):
            res = future.result()
            if res:
                results.extend(res)
    
    # Save results to a CSV file
    results_df = pd.DataFrame(results, columns=['Ticker', 'Complexity', 'RMSE', 'R2', 'Sharpe_Ratio'])
    results_df.to_csv('model_results.csv', index=False)

    # Filter and sort models based on Sharpe Ratio, RMSE, and R2
    filtered_results = results_df[results_df['R2'] >= 0.5]
    filtered_results = filtered_results.sort_values(by=['Sharpe_Ratio', 'RMSE'], ascending=[False, True])

    top_10_models = filtered_results.head(10)
    top_10_models.to_csv('top_10_models.csv', index=False)

main()


[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed


[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed


[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed


[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  1 of 1 completed


C:\Users\Asad\AppData\Local\Temp\ipykernel_16828\436517455.py:41: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data['Volume'] = data['Volume'].replace(0, np.nan).fillna(method='ffill')
C:\Users\Asad\AppData\Local\Temp\ipykernel_16828\436517455.py:41: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data['Volume'] = data['Volume'].replace(0, np.nan).fillna(method='ffill')
C:\Users\Asad\AppData\Local\Temp\ipykernel_16828\436517455.py:41: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data['Volume'] = data['Volume'].replace(0, np.nan).fillna(method='ffill')
C:\Users\Asad\AppData\Local\Temp\ipykernel_16828\436517455.py:41: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill()

5/5 ━━━━━━━━━━━━━━━━━━━━ 4s 548ms/step


... (95 outputs left out)

5/5 ━━━━━━━━━━━━━━━━━━━━ 14s 2s/step 


c:\Users\Asad\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


5/5 ━━━━━━━━━━━━━━━━━━━━ 13s 2s/step


c:\Users\Asad\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


# Trade

In [19]:
import yfinance as yf
import pandas as pd
import numpy as np
import tensorflow as tf
import os
import pickle
from sklearn.preprocessing import StandardScaler

# Constants
start_date_for_sequences = '2023-12-01'  # Ensure enough data for sequences
april_2024_start = '2024-04-01'
june_2024_end = '2024-06-30'
slippage_percentage = 0.001  # 0.1% slippage
transaction_fee_percentage = 0.001  # 0.1% transaction fee per trade
max_position_size = 100  # Maximum number of shares to hold in any position
max_trade_size = 10  # Maximum number of shares per trade

# Load the top models from top_10_models.csv and select the best model for each ticker
top_10_models = pd.read_csv('top_10_models.csv')
best_models = top_10_models.loc[top_10_models.groupby('Ticker')['Sharpe_Ratio'].idxmax()]

# Create directories if they don't exist
os.makedirs('backtest_results', exist_ok=True)
os.makedirs('plots', exist_ok=True)

def download_data(ticker, start, end):
    try:
        print(f"Downloading data for {ticker} from {start} to {end}")
        data = yf.download(ticker, start=start, end=end)
        if data.empty:
            print(f"No data found for {ticker}")
            return None
        return data
    except Exception as e:
        print(f"Error downloading data for {ticker}: {e}")
        return None

def load_model_and_scaler(ticker, complexity):
    model_path = f'models/{ticker}_complexity_{complexity}.keras'
    scaler_path = f'scalers/{ticker}_scaler_{complexity}.pkl'

    try:
        model = tf.keras.models.load_model(model_path)
        with open(scaler_path, 'rb') as f:
            scaler = pickle.load(f)
        return model, scaler
    except Exception as e:
        print(f"Error loading model/scaler for {ticker}, complexity {complexity}: {e}")
        return None, None

def calculate_rsi(series, period=14):
    delta = series.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def calculate_macd(series, fast_period=12, slow_period=26, signal_period=9):
    fast_ema = series.ewm(span=fast_period, min_periods=fast_period).mean()
    slow_ema = series.ewm(span=slow_period, min_periods=slow_period).mean()
    macd = fast_ema - slow_ema
    signal = macd.ewm(span=signal_period, min_periods=signal_period).mean()
    return macd, signal

def preprocess_data(data):
    try:
        data['Volume'] = data['Volume'].replace(0, np.nan).ffill()
        data['RSI'] = calculate_rsi(data['Close'])
        data['MACD'], data['MACD_Signal'] = calculate_macd(data['Close'])
        data = data[['Close', 'Volume', 'RSI', 'MACD', 'MACD_Signal']].dropna()
        return data
    except Exception as e:
        print(f"Error preprocessing data: {e}")
        return None

def create_sequences(data, time_steps=20):
    sequences, targets = [], []
    try:
        for i in range(len(data) - time_steps):
            sequences.append(data[i:i+time_steps])
            targets.append(data[i+time_steps, 0])
    except Exception as e:
        print(f"Error creating sequences: {e}")
    return np.array(sequences), np.array(targets)

def predict_prices(model, scaler, data):
    try:
        data_scaled = scaler.transform(data)
        sequences = [data_scaled[i:i+20] for i in range(len(data_scaled) - 20)]
        sequences = np.array(sequences)
        predictions_scaled = model.predict(sequences)
        # Inverse transform the predictions to original scale
        predictions = scaler.inverse_transform(
            np.concatenate([predictions_scaled, np.zeros((predictions_scaled.shape[0], data.shape[1] - 1))], axis=1)
        )[:, 0]
        return predictions
    except Exception as e:
        print(f"Error predicting prices: {e}")
        return None

def backtest_strategy(ticker, predictions, actual_data):
    initial_balance = 10000  # Starting with $10,000
    balance = initial_balance
    shares = 0
    short_shares = 0
    trade_log = []

    for i in range(len(predictions)):
        actual_price = actual_data['Close'].values[i]
        predicted_price = predictions[i]
        
        # Apply slippage
        buy_price = actual_price * (1 + slippage_percentage)
        sell_price = actual_price * (1 - slippage_percentage)

        # Ensure predictions are within a reasonable range
        if predicted_price <= 0 or np.isnan(predicted_price):
            continue

        # Buy signal: Buy if predicted price > actual price and we have cash
        if predicted_price > actual_price and balance >= buy_price and short_shares == 0:
            shares_bought = min(balance // buy_price, max_trade_size, max_position_size - shares)
            if shares_bought > 0:
                total_cost = shares_bought * buy_price
                transaction_fee = total_cost * transaction_fee_percentage
                balance -= (total_cost + transaction_fee)
                shares += shares_bought
                trade_log.append((actual_data.index[i], 'Buy', shares_bought, buy_price, balance, shares, short_shares))
                print(f"Buying {shares_bought} shares of {ticker} at {buy_price}")
        
        # Sell signal: Sell if predicted price < actual price and we have shares
        elif predicted_price < actual_price and shares > 0:
            shares_sold = min(shares, max_trade_size)
            if shares_sold > 0:
                total_revenue = shares_sold * sell_price
                transaction_fee = total_revenue * transaction_fee_percentage
                balance += (total_revenue - transaction_fee)
                shares -= shares_sold
                trade_log.append((actual_data.index[i], 'Sell', shares_sold, sell_price, balance, shares, short_shares))
                print(f"Selling {shares_sold} shares of {ticker} at {sell_price}")
        
        # Short sell signal: Short sell if predicted price < actual price and we have no long shares
        elif predicted_price < actual_price and shares == 0:
            short_shares_sold = min(balance // sell_price, max_trade_size, max_position_size - short_shares)
            if short_shares_sold > 0:
                total_revenue = short_shares_sold * sell_price
                transaction_fee = total_revenue * transaction_fee_percentage
                balance += (total_revenue - transaction_fee)
                short_shares += short_shares_sold
                trade_log.append((actual_data.index[i], 'Short Sell', short_shares_sold, sell_price, balance, shares, short_shares))
                print(f"Short selling {short_shares_sold} shares of {ticker} at {sell_price}")
        
        # Buy to cover signal: Buy back short shares if predicted price > actual price and we have short shares
        elif predicted_price > actual_price and short_shares > 0:
            shares_bought = min(short_shares, max_trade_size)
            if shares_bought > 0:
                total_cost = shares_bought * buy_price
                transaction_fee = total_cost * transaction_fee_percentage
                balance -= (total_cost + transaction_fee)
                short_shares -= shares_bought
                trade_log.append((actual_data.index[i], 'Buy to Cover', shares_bought, buy_price, balance, shares, short_shares))
                print(f"Buying to cover {shares_bought} shares of {ticker} at {buy_price}")

    final_balance = balance + shares * actual_data['Close'].values[-1] - short_shares * actual_data['Close'].values[-1]
    return initial_balance, final_balance, trade_log

def main():
    backtest_results = []

    for index, row in best_models.iterrows():
        ticker = row['Ticker']
        complexity = row['Complexity']

        # Download data from December 2023 to June 2024 for sequence creation
        data = download_data(ticker, start_date_for_sequences, june_2024_end)
        if data is None:
            continue

        data = preprocess_data(data)
        if data is None:
            continue

        model, scaler = load_model_and_scaler(ticker, complexity)
        if model is None or scaler is None:
            continue

        # Generate sequences from the entire dataset
        sequences, _ = create_sequences(data.values, time_steps=20)

        predictions_scaled = model.predict(sequences)
        predictions = scaler.inverse_transform(
            np.concatenate([predictions_scaled, np.zeros((predictions_scaled.shape[0], data.shape[1] - 1))], axis=1)
        )[:, 0]

        # Use only the predictions corresponding to April, May, and June 2024 for backtesting
        april_may_june_data = data.loc[april_2024_start:june_2024_end]
        if april_may_june_data.empty:
            print(f"No April to June 2024 data for {ticker}")
            continue

        # Align predictions with the actual data length
        required_length = len(april_may_june_data)
        start_index = len(predictions) - required_length
        predictions_for_april_may_june = predictions[start_index:]

        if len(predictions_for_april_may_june) != required_length:
            print(f"Prediction length mismatch for {ticker}")
            print(f"Predictions length: {len(predictions_for_april_may_june)}")
            print(f"Actual data length: {len(april_may_june_data)}")
            continue

        print(f"Predictions for {ticker}: {predictions_for_april_may_june}")
        print(f"Actual prices for {ticker}: {april_may_june_data['Close'].values}")

        initial_balance, final_balance, trade_log = backtest_strategy(ticker, predictions_for_april_may_june, april_may_june_data)

        percent_return = ((final_balance - initial_balance) / initial_balance) * 100

        backtest_results.append({
            'Ticker': ticker,
            'Initial Balance': initial_balance,
            'Final Balance': final_balance,
            'Profit/Loss': final_balance - initial_balance,
            'Percent Return': percent_return
        })

        # Save the trade log for each ticker
        trade_log_df = pd.DataFrame(trade_log, columns=['Date', 'Action', 'Shares', 'Price', 'Balance', 'Total Shares', 'Short Shares'])
        trade_log_df.to_csv(f'backtest_results/{ticker}_trade_log.csv', index=False)

    # Save the summary of backtest results
    backtest_results_df = pd.DataFrame(backtest_results)
    backtest_results_df.to_csv('backtest_results/backtest_summary.csv', index=False)

    print(backtest_results_df)

main()


[*********************100%%**********************]  1 of 1 completed

3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 336ms/step


[*********************100%%**********************]  1 of 1 completed

Predictions for AAPL: [106.45892891 106.45892891 106.45892612 106.45892612 106.45892891
 106.45892891 106.45892891 106.45892891 106.45892891 106.45892891
 106.45892891 106.45892891 106.45892891 106.45892891 106.45892891
 106.45892891 106.45892891 106.45892891 106.45892891 106.45892891
 106.45892891 106.45892891 106.45892891 106.45892891 106.45892891
 106.45892891 106.45892891 106.45892891 106.45892891 106.45892891
 106.45892891 106.45892891 106.45892891 106.45892891 106.45892612
 106.45892612 106.45892891 106.45892891 106.45892891 106.45892891
 106.45892891 106.45892891 106.45892891 106.45892891 106.45892891
 106.45892891 106.45892891 106.45892891 106.45892891 106.45892891
 106.45892891 106.45892891 106.45892891 106.45892891 106.45892891
 106.45892891 106.45892891 106.45892891 106.45892891 106.45892891
 106.45892612 106.45892612 106.45892612]
Actual prices for AAPL: [170.02999878 168.83999634 169.6499939  168.82000732 169.58000183
 168.44999695 169.66999817 167.77999878 175.03999329 17

3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 321ms/step


[*********************100%%**********************]  1 of 1 completed

Predictions for MCD: [203.99132225 203.99132225 203.99131815 203.99131815 203.99132225
 203.99132225 203.99132225 203.99132225 203.99132225 203.99132225
 203.99132225 203.99132225 203.99132225 203.99132225 203.99132225
 203.99132225 203.99132225 203.99132225 203.99132225 203.99132225
 203.99132225 203.99132225 203.99132225 203.99132225 203.99132225
 203.99132225 203.99132225 203.99132225 203.99132225 203.99132225
 203.99132225 203.99132225 203.99132225 203.99132225 203.99131815
 203.99131815 203.99132225 203.99132225 203.99132225 203.99132225
 203.99132225 203.99132225 203.99132225 203.99132225 203.99132225
 203.99132225 203.99132225 203.99132225 203.99132225 203.99132225
 203.99132225 203.99132225 203.99132225 203.99132225 203.99132225
 203.99132225 203.99132225 203.99132225 203.99132225 203.99132225
 203.99131815 203.99131815 203.99131815]
Actual prices for MCD: [280.22000122 277.73999023 275.54000854 270.08999634 266.69000244
 267.55999756 269.44000244 268.67001343 268.61999512 267.

3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 163ms/step


[*********************100%%**********************]  1 of 1 completed

Predictions for MRK: [106.50374752 106.50374752 106.5037496  106.5037496  106.50374752
 106.50374752 106.50374752 106.50374752 106.50374752 106.50374752
 106.50374752 106.50374752 106.50374752 106.50374752 106.50374752
 106.50374752 106.50374752 106.50374752 106.50374752 106.50374752
 106.50374752 106.50374752 106.50374752 106.50374752 106.50374752
 106.50374752 106.50374752 106.50374752 106.50374752 106.50374752
 106.50374752 106.50374752 106.50374752 106.50374752 106.5037496
 106.5037496  106.50374752 106.50374752 106.50374752 106.50374752
 106.50374752 106.50374752 106.50374752 106.50374752 106.50374752
 106.50374752 106.50374752 106.50374752 106.50374752 106.50374752
 106.50374752 106.50374752 106.50374752 106.50374752 106.50374752
 106.50374752 106.50374752 106.50374752 106.50374752 106.50374752
 106.50374752 106.50374752 106.50374856]
Actual prices for MRK: [130.99000549 130.3500061  129.8999939  127.66000366 127.98999786
 126.55999756 126.70999908 126.75       126.15000153 125.7

3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 229ms/step
Predictions for UNH: [425.2356308  425.2356308  425.23562604 425.23562604 425.2356308
 425.2356308  425.2356308  425.2356308  425.2356308  425.2356308
 425.2356308  425.2356308  425.2356308  425.2356308  425.2356308
 425.2356308  425.2356308  425.2356308  425.2356308  425.2356308
 425.2356308  425.2356308  425.2356308  425.2356308  425.2356308
 425.2356308  425.2356308  425.2356308  425.2356308  425.2356308
 425.2356308  425.2356308  425.2356308  425.2356308  425.23562604
 425.23562604 425.2356308  425.2356308  425.2356308  425.2356308
 425.2356308  425.2356308  425.2356308  425.2356308  425.2356308
 425.2356308  425.2356308  425.2356308  425.2356308  425.2356308
 425.2356308  425.2356308  425.2356308  425.2356308  425.2356308
 425.2356308  425.2356308  425.2356308  425.2356308  425.2356308
 425.2356308  425.2356308  425.2356308 ]
Actual prices for UNH: [489.70001221 458.14001465 459.73999023 455.38000488 455.73999023
 456.         459.72000122 45